[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/09_nlp/36_topic_modeling_stream.ipynb)

> 📎 **Optional module — reference style.** Module 9 is optional. Like the appendices, these notebooks are written as a demo / reference: they focus on *seeing* a library at work rather than interactive exercises. Each notebook is built to run end-to-end *without* the optional library — it falls back to a small built-in stand-in — so you can read and run it offline. Install the optional library (see **Install** below) to swap the stand-in for the real thing.

---

# 📓 Notebook 36 — Topic Modeling with STREAM

> **Module:** 12 · Optional · **Type:** Reference · **Estimated time:** 45–75 min · **Difficulty:** Intermediate

In Notebook 35 we used **BERTopic** — one excellent, opinionated pipeline (embed → reduce → cluster → c-TF-IDF). **STREAM** ([`stream-topic`](https://github.com/AnFreTh/STREAM)) takes a different stance. Instead of *one* model, it is a **unified API over many topic models**:

- **Classical** models — `LDA`, `NMF`.
- **Neural** models — `ETM`, `CTM`, `ProdLDA`, `NeuralLDA`, `NSTM`, `TNTM`, `WordCluTM`.
- **Clustering / embedding** models — `KmeansTM`, `CEDC`, `DCTE`, `SomTM`, `CBC`.

Every one of these is trained the same way: `model.fit(dataset, n_topics=...)` then `model.get_topics()`. Swapping `KmeansTM()` for `ProdLDA()` is a **one-line change** — which makes STREAM ideal for *honest model comparison* rather than betting your analysis on a single algorithm.

On top of the models, STREAM ships:

- a **strong evaluation suite** — embedding-based coherence/diversity metrics (`ISIM`, `INT`, `ISH`, `Expressivity`) that correlate with human judgement, plus the classic word-cooccurrence `NPMI`;
- **visualizations** (`visualize_topic_model`, `visualize_topics`); and
- **downstream tasks** — feed the learned topic representation into a **Neural Additive Model** (`DownstreamModel`) for interpretable regression/classification.

**Business framing.** Whether you are mining thousands of customer-feedback tickets, clustering a news/research corpus, or summarising survey free-text, the hard question is rarely "can I run *a* topic model?" — it is "*which* model, and is it any good?" STREAM is built to answer exactly that.

## 🎯 Learning objectives

By the end you will be able to:

1. Explain what STREAM adds on top of a single pipeline like BERTopic — a **unified interface** across classical, neural and clustering topic models.
2. Load and preprocess a corpus with `TMDataset` and fit a model with the canonical `fit` / `get_topics` flow.
3. Swap model families (`KmeansTM` ↔ `ProdLDA`) with no other code change, and know when to reach for **probabilistic** (`get_beta`/`get_theta`) vs **clustering** models.
4. Evaluate topics with embedding metrics (`ISIM`/`INT`/`ISH`/`Expressivity`) and classic `NPMI`, and read the difference.
5. Search the number of topics with `optimize_and_fit`, and understand the `DownstreamModel` (NAM) idea for prediction.

## ✅ Prerequisites

- **NB 35 — Topic Modeling with BERTopic** (recommended): the embed → cluster → label intuition carries over directly.
- **NB 23 — Embeddings**: STREAM's best metrics and several models are embedding-based.
- Comfort with `numpy`, `pandas`, `scikit-learn`, and a little `matplotlib`/`seaborn`.

## 📦 Install

```bash
pip install stream-topic
```

> STREAM also pulls in `lightning` (for the downstream NAM) and sentence-transformers for the embedding metrics. **You do not need any of this to follow along** — every cell below has an offline stand-in. If `stream-topic` is not importable, the notebook quietly uses a small `scikit-learn` topic model and a NumPy NPMI proxy instead.

## 1. Setup & offline smoke-test

We first try to import `stream_topic`. The boolean `HAS_STREAM` then guards every STREAM-specific cell: if the real package is present we run the real API; otherwise we fall back to a lightweight `scikit-learn` stand-in so the notebook still executes top-to-bottom.

This *offline-first* pattern is worth internalising as a practitioner: heavy ML libraries are flaky to install in CI, on locked-down corporate laptops, or in a graded environment. Writing code that **degrades gracefully** keeps your teaching material — and your demos — reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# --- Offline smoke-test: is the real STREAM library available? ---
try:
    import stream_topic  # noqa: F401
    HAS_STREAM = True
    print("stream-topic is installed -> running the REAL STREAM API.")
except Exception as e:  # ImportError, but be permissive
    HAS_STREAM = False
    print("stream-topic NOT found -> using the offline scikit-learn stand-in.")
    print(f"   (reason: {type(e).__name__})")

print(f"HAS_STREAM = {HAS_STREAM}")

## 2. A small business corpus

To keep the notebook offline and fast we define ~36 short documents inline: a mix of **product reviews**, **support tickets** and **finance/news** snippets. Three latent themes are baked in — *shipping/delivery*, *app/login/technical*, and *price/billing* — so we can sanity-check whatever model we fit.

In a real project this would be a column of a `DataFrame` loaded from a CSV, a database, or the Zendesk/Intercom API.

In [ ]:
DOCS = [
    # --- shipping / delivery ---
    "My order arrived two days late and the box was damaged in transit.",
    "Fast shipping, the package was delivered earlier than the estimate.",
    "The courier left my parcel in the rain and the contents were soaked.",
    "Delivery tracking never updated so I had no idea where my package was.",
    "Great experience, free shipping and the item shipped the same day.",
    "Package was returned to sender because the delivery address was wrong.",
    "Shipment took three weeks to clear customs, far too slow.",
    "The delivery driver was friendly and the parcel arrived intact.",
    "Still waiting on my refund for a package that was never delivered.",
    "Order shipped quickly but arrived with the wrong items inside.",
    "Express delivery was worth it, the parcel came overnight.",
    "Tracking said delivered but the package was nowhere to be found.",
    # --- app / login / technical ---
    "The mobile app keeps crashing every time I open the dashboard.",
    "I cannot log in, the password reset email never arrives.",
    "After the latest update the app freezes on the loading screen.",
    "Login fails with an error code and support has not responded.",
    "The web portal is slow and the page times out when I save settings.",
    "Two-factor authentication is broken, the code never sends to my phone.",
    "App notifications stopped working after I reinstalled the software.",
    "The dashboard shows a blank screen and the data never loads.",
    "Sync between the desktop and mobile app fails constantly.",
    "I get logged out randomly and have to reset my password again.",
    "The new interface is buggy and the search feature returns nothing.",
    "Crash on startup on Android, the app worked fine last week.",
    # --- price / billing / finance ---
    "I was charged twice for the same subscription this month.",
    "The price went up without any notice on my monthly invoice.",
    "Cancelling my plan still resulted in a billing charge on my card.",
    "Great value for the money, the premium plan is fairly priced.",
    "My invoice does not match the quoted price, I was overcharged.",
    "The refund for the cancelled order has not hit my account yet.",
    "Hidden fees made the final cost much higher than advertised.",
    "Affordable pricing and a clear monthly bill, no surprises.",
    "They billed my credit card after the free trial without warning.",
    "The discount code did not apply so I paid full price at checkout.",
    "Subscription cost is reasonable but the payment page rejects my card.",
    "Disputed an incorrect charge and the billing team fixed the invoice.",
]

corpus = pd.Series(DOCS, name="text")
print(f"{len(corpus)} documents.")
corpus.head(3)

## 3. The offline stand-in

Before touching STREAM we build a tiny **stand-in topic model** out of `scikit-learn`: a `TfidfVectorizer` feeding an `NMF` factorisation. NMF on a TF-IDF matrix is a perfectly legitimate (classical) topic model — in fact `NMF` is one of the models STREAM exposes — so this is not a toy: it is a faithful, dependency-light version of the same idea.

We wrap it in a small class whose method names *mirror* the STREAM API (`fit`, `get_topics`, `get_beta`, `get_theta`). That way the rest of the notebook can call the same methods whether or not the real library is installed.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

class StandInTopicModel:
    """A minimal NMF-over-TF-IDF topic model with a STREAM-like API."""

    def __init__(self, n_topics=3, top_n=8, random_state=42):
        self.n_topics = n_topics
        self.top_n = top_n
        self.random_state = random_state

    def fit(self, docs):
        self.vectorizer_ = TfidfVectorizer(
            stop_words="english", min_df=2, ngram_range=(1, 1)
        )
        X = self.vectorizer_.fit_transform(docs)
        self.vocab_ = np.array(self.vectorizer_.get_feature_names_out())
        self.nmf_ = NMF(
            n_components=self.n_topics, init="nndsvda",
            random_state=self.random_state, max_iter=500,
        )
        self.theta_ = self.nmf_.fit_transform(X)   # docs x topics
        self.beta_ = self.nmf_.components_          # topics x words
        self.X_ = X
        return self

    def get_topics(self):
        """Return {topic_id: [top words]} — mirrors STREAM's get_topics()."""
        topics = {}
        for k in range(self.n_topics):
            order = np.argsort(self.beta_[k])[::-1][: self.top_n]
            topics[k] = list(self.vocab_[order])
        return topics

    def get_beta(self):
        """Topic-word weight matrix (topics x vocab)."""
        return self.beta_

    def get_theta(self):
        """Document-topic weight matrix (docs x topics)."""
        return self.theta_

print("StandInTopicModel defined.")

## 4. Fitting a model — the canonical STREAM flow

Here is the **real** STREAM API you would write with `stream-topic` installed:

```python
from stream_topic.models import KmeansTM
from stream_topic.utils import TMDataset

dataset = TMDataset()
dataset.fetch_dataset("BBC_News")          # or 20NewsGroups, Reuters, Spotify, Reddit_GME, Poliblogs ...
dataset.preprocess(model_type="KmeansTM")  # tokenise, clean, build embeddings

model = KmeansTM()
model.fit(dataset, n_topics=20)
topics = model.get_topics()                 # {topic_id: [top words]}
```

Two things to notice:

1. **The data lives in a `TMDataset`.** STREAM owns preprocessing — tokenisation, cleaning, *and* the document embeddings the neural/clustering models need. You call `preprocess(model_type=...)` so it can prepare exactly what that model family expects.
2. **`fit` + `get_topics` is universal.** That same two-line pattern works for `LDA`, `ProdLDA`, `ETM`, `CEDC`, ... — which is the whole point.

The cell below runs the real flow if STREAM is present, and otherwise fits our stand-in on the inline corpus. Either branch ends with a `topics` dict of `{id: [words]}`.

In [ ]:
N_TOPICS = 3  # we baked in 3 themes; raise this on a real corpus

if HAS_STREAM:
    from stream_topic.models import KmeansTM
    from stream_topic.utils import TMDataset

    # On a real machine you would fetch a bundled corpus, e.g. BBC_News.
    # To stay self-contained we load our inline DOCS into a TMDataset.
    dataset = TMDataset()
    try:
        dataset.create_load_save_dataset(
            data=pd.DataFrame({"text": DOCS}),
            dataset_name="nb44_demo",
            save_dir="./nb44_data",
            doc_column="text",
        )
    except Exception:
        dataset.fetch_dataset("BBC_News")  # fallback to a bundled corpus
    dataset.preprocess(model_type="KmeansTM")

    model = KmeansTM()
    model.fit(dataset, n_topics=N_TOPICS)
    topics = model.get_topics()
else:
    model = StandInTopicModel(n_topics=N_TOPICS, top_n=8).fit(DOCS)
    topics = model.get_topics()

for tid, words in topics.items():
    print(f"Topic {tid}: {', '.join(words[:8])}")

## 5. The killer feature — swap the model, change nothing else

This is STREAM's central promise. To compare a **clustering** model against a **probabilistic neural** model, you change *one constructor*:

```python
# Clustering / embedding model
model = KmeansTM()
model.fit(dataset, n_topics=20)

# Probabilistic neural model — identical training code
from stream_topic.models import ProdLDA
model = ProdLDA()
model.fit(dataset, n_topics=20)
```

### Two families, two mental models

| | **Probabilistic** (`LDA`, `ProdLDA`, `NeuralLDA`, `ETM`, `CTM`) | **Clustering / embedding** (`KmeansTM`, `CEDC`, `DCTE`, `SomTM`) |
|---|---|---|
| What a topic *is* | a probability distribution over the vocabulary | a cluster of documents in embedding space |
| Topic–word weights | `get_beta()` → P(word \| topic) | weights derived from cluster centroids |
| Document mixture | `get_theta()` → P(topic \| doc); a doc is a *mix* of topics | typically a hard/soft assignment to a cluster |
| Strength | principled, soft mixed-membership | leverages pretrained embeddings, great short-text |

Below we demonstrate the swap. With STREAM installed it really fits `ProdLDA`; offline it re-uses the stand-in. Note our stand-in is **NMF** — a *classical matrix factorisation*, **not** a probabilistic model (its `beta`/`theta` are nonnegative weights, not true probabilities). It can't reproduce ProdLDA's behaviour; it simply exposes the same `get_beta`/`get_theta` surface so the comparison still prints offline.

In [ ]:
if HAS_STREAM:
    from stream_topic.models import ProdLDA
    prob_model = ProdLDA()          # <-- the ONLY line that changed
    prob_model.fit(dataset, n_topics=N_TOPICS)
    prob_topics = prob_model.get_topics()
else:
    # Our stand-in is a classical NMF, not a probabilistic model -- but it
    # exposes the same get_beta / get_theta surface, so we reuse it here to
    # illustrate the swap mechanics (and those matrices) offline.
    prob_model = model
    prob_topics = topics

print("ProdLDA-style topics:")
for tid, words in prob_topics.items():
    print(f"  Topic {tid}: {', '.join(words[:8])}")

## 6. `get_beta` and `get_theta` — what a probabilistic model gives you

Probabilistic models expose two matrices that are gold for downstream analysis:

- **`get_beta()`** → topic × vocabulary. Row *k* is the weight of each word under topic *k* (a true P(word | topic) for probabilistic models; just a nonnegative weight for our NMF stand-in). This is what produces the "top words" lists.
- **`get_theta()`** → document × topic. Row *i* tells you how much document *i* belongs to each topic — a soft, mixed-membership profile. This is the feature vector you would feed into a classifier, a dashboard, or the `DownstreamModel` later.

Let's inspect both for our model and visualise the document–topic mixture as a heatmap.

In [ ]:
beta = prob_model.get_beta()    # topics x vocab
theta = prob_model.get_theta()  # docs x topics

# Normalise theta rows to read as 'share of each topic per document'.
theta_norm = theta / (theta.sum(axis=1, keepdims=True) + 1e-12)

print(f"beta  shape (topics x vocab): {beta.shape}")
print(f"theta shape (docs x topics):  {theta.shape}")

fig, ax = plt.subplots(figsize=(6, 9))
sns.heatmap(
    theta_norm, cmap="viridis", cbar_kws={"label": "topic share"},
    xticklabels=[f"T{k}" for k in range(theta.shape[1])], ax=ax,
)
ax.set_title("Document-topic mixture (theta)")
ax.set_xlabel("topic"); ax.set_ylabel("document index")
plt.tight_layout(); plt.show()

dominant = theta_norm.argmax(axis=1)
print("\nDominant topic per (first 8) documents:")
for i in range(8):
    print(f"  doc {i:2d} -> T{dominant[i]} | {DOCS[i][:55]}...")

## 7. Evaluation — is this topic model any good?

Top-words lists *look* convincing even when a model is bad, so we quantify quality. STREAM groups its metrics into two ideas:

**Embedding-based metrics** (correlate well with human judgement; require sentence embeddings):

- **`ISIM`** — *Intruder Similarity Metric.* Average embedding similarity of a topic's words to an "intruder" word; lower means the topic is more cohesive (the intruder stands out).
- **`INT`** — *Intruder Accuracy.* How reliably an intruder word can be detected — high = well-separated topics.
- **`ISH`** — *Intruder Shift.* How much the topic centroid shifts when an intruder is injected.
- **`Expressivity`** — how distinct / non-redundant topics are from generic "meaningless" words (a diversity-flavoured measure).

**Word-cooccurrence metric** (classic, no embeddings needed):

- **`NPMI`** — *Normalised Pointwise Mutual Information.* For each pair of top words, does seeing one make the other more likely *in the corpus*? Averaged over pairs and topics. The long-standing standard for automatic coherence.

The real STREAM call is uniform across metrics:

```python
from stream_topic.metrics import ISIM, INT, ISH, Expressivity, NPMI
metric = ISIM()
metric.score(topics)             # one corpus-level number
metric.score_per_topic(topics)   # one number per topic
```

Offline we cannot compute embedding metrics, but we *can* implement an honest **NPMI proxy** in NumPy — which is exactly the classic coherence measure.

In [ ]:
def npmi_coherence(topics, docs, top_n=8, eps=1e-12):
    """Offline NPMI coherence proxy.

    For each topic, average the normalised PMI over all pairs of its
    top-N words, using document co-occurrence counts. Returns the
    corpus mean and a per-topic dict -- mirroring STREAM's
    score() / score_per_topic() split.
    """
    # Build a binary word-in-doc presence table over the union of top words.
    vocab = sorted({w for words in topics.values() for w in words[:top_n]})
    idx = {w: j for j, w in enumerate(vocab)}
    tokenised = [set(d.lower().split()) for d in docs]
    # strip simple punctuation
    tokenised = [{t.strip(".,!?;:'\"") for t in toks} for toks in tokenised]
    N = len(docs)
    present = np.zeros((N, len(vocab)), dtype=float)
    for i, toks in enumerate(tokenised):
        for w in vocab:
            if w in toks:
                present[i, idx[w]] = 1.0

    p = present.mean(axis=0)                       # P(word)
    co = (present.T @ present) / N                 # P(word_i, word_j)

    per_topic = {}
    for tid, words in topics.items():
        ws = [w for w in words[:top_n] if w in idx]
        scores = []
        for a in range(len(ws)):
            for b in range(a + 1, len(ws)):
                ia, ib = idx[ws[a]], idx[ws[b]]
                pij = co[ia, ib]
                if pij <= 0:
                    scores.append(-1.0)            # never co-occur
                    continue
                pmi = np.log((pij + eps) / (p[ia] * p[ib] + eps))
                npmi = pmi / (-np.log(pij + eps))  # normalise to [-1, 1]
                scores.append(npmi)
        per_topic[tid] = float(np.mean(scores)) if scores else 0.0
    corpus = float(np.mean(list(per_topic.values())))
    return corpus, per_topic


if HAS_STREAM:
    from stream_topic.metrics import NPMI
    npmi = NPMI(dataset)
    corpus_npmi = npmi.score(topics)
    per_topic_npmi = npmi.score_per_topic(topics)
    print("NPMI via STREAM:")
else:
    corpus_npmi, per_topic_npmi = npmi_coherence(topics, DOCS, top_n=8)
    print("NPMI via offline NumPy proxy:")

print(f"  corpus NPMI = {corpus_npmi:+.3f}  (higher = more coherent)")
for tid, s in (per_topic_npmi.items() if isinstance(per_topic_npmi, dict)
               else enumerate(per_topic_npmi)):
    print(f"  topic {tid}: {s:+.3f}")

### Embedding metrics in practice

With `stream-topic` installed, the embedding metrics read identically — only the class name changes:

```python
from stream_topic.metrics import ISIM, INT, ISH, Expressivity

for Metric in (ISIM, INT, ISH, Expressivity):
    m = Metric()                  # embedding metrics need no dataset
    print(Metric.__name__, m.score(topics))
```

(The embedding metrics embed the *topic words* with a sentence-transformer, so they take no dataset — unlike `NPMI` above, which needs the corpus to count word co-occurrences and is therefore constructed as `NPMI(dataset)`.)

**How to read them together.** NPMI is cheap and language-agnostic but rewards words that simply co-occur (it can be fooled by boilerplate). The embedding metrics ask a harder, more human question — *do these words mean similar things?* — and catch "junk" topics that NPMI misses. A senior practitioner reports **both** and treats large disagreements as a signal to inspect the topic by hand. Below we visualise our per-topic NPMI.

In [ ]:
tids = list(per_topic_npmi.keys()) if isinstance(per_topic_npmi, dict) else list(range(len(per_topic_npmi)))
vals = list(per_topic_npmi.values()) if isinstance(per_topic_npmi, dict) else list(per_topic_npmi)
labels = [", ".join(topics[t][:3]) for t in tids]

fig, ax = plt.subplots(figsize=(8, 0.7 * len(tids) + 1))
colors = ["#2a9d8f" if v >= 0 else "#e76f51" for v in vals]
ax.barh(range(len(tids)), vals, color=colors)
ax.set_yticks(range(len(tids)))
ax.set_yticklabels([f"T{t}: {l}" for t, l in zip(tids, labels)])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("per-topic NPMI coherence")
ax.set_title("Topic coherence (higher is better)")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 8. How many topics? `optimize_and_fit`

Choosing `n_topics` by eye is fragile. STREAM can search it for you, refitting across a range and selecting by an information criterion:

```python
model.optimize_and_fit(
    dataset, min_topics=2, max_topics=20,
    criterion="aic",   # also 'bic'
    n_trials=20,
)
```

This is the same idea as a coherence/elbow sweep you might do by hand, but built in and consistent across model families. Offline we emulate the *sweep* by refitting our stand-in for several values of *k* and tracking NPMI, then pick the peak.

> ⚠️ **Don't expect it to recover the 3 themes we planted.** On only 36 tiny documents, word-cooccurrence NPMI is noisy and the corpus values are all slightly negative — here the peak lands at a *smaller* `k` than 3, not because 3 is wrong but because a single cheap metric on a toy corpus can't see it. That *is* the lesson: treat automatic `k`-selection as a starting point to **inspect**, not gospel. On real data you would sweep a wider range and cross-check against the embedding metrics and a human read. (The real `optimize_and_fit` also uses AIC/BIC, not NPMI — a different objective again.)

In [ ]:
if HAS_STREAM:
    # Real STREAM: one call searches and refits the model in place.
    model.optimize_and_fit(dataset, min_topics=2, max_topics=8,
                           criterion="aic", n_trials=8)
    best_k = len(model.get_topics())
    ks, coh = [best_k], [np.nan]
    print(f"STREAM optimize_and_fit selected k = {best_k}")
else:
    ks = list(range(2, 9))
    coh = []
    for k in ks:
        m_k = StandInTopicModel(n_topics=k, top_n=8).fit(DOCS)
        c, _ = npmi_coherence(m_k.get_topics(), DOCS, top_n=8)
        coh.append(c)
    best_k = ks[int(np.argmax(coh))]
    print(f"Offline sweep selected k = {best_k} (peak NPMI)")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ks, coh, marker="o")
    ax.axvline(best_k, color="#e76f51", ls="--", label=f"best k = {best_k}")
    ax.set_xlabel("number of topics (k)")
    ax.set_ylabel("corpus NPMI")
    ax.set_title("Topic-count sweep (offline emulation of optimize_and_fit)")
    ax.legend(); plt.tight_layout(); plt.show()

## 9. Visualization

STREAM ships interactive Plotly/Dash visualisations that launch a small local server:

```python
from stream_topic.visuals import visualize_topic_model, visualize_topics

visualize_topic_model(model, reduce_first=True, port=8051)  # interactive map of topics
visualize_topics(model)                                     # topic word barcharts
```

`visualize_topic_model` projects topics into 2-D (set `reduce_first=True` to reduce embeddings before plotting) so you can *see* which topics sit close together — invaluable for spotting near-duplicate topics that should be merged.

Because that needs a running server (and the library), we render an **offline static analogue**: a 2-D scatter of documents coloured by dominant topic, using PCA over the TF-IDF matrix. Same diagnostic intent, no server required.

In [ ]:
if HAS_STREAM:
    print("Run locally to launch the interactive view:")
    print("    from stream_topic.visuals import visualize_topic_model")
    print("    visualize_topic_model(model, reduce_first=True, port=8051)")

# Offline static analogue: PCA scatter of documents, coloured by topic.
from sklearn.decomposition import PCA

standin = model if not HAS_STREAM else StandInTopicModel(N_TOPICS).fit(DOCS)
X = standin.X_.toarray()
coords = PCA(n_components=2, random_state=0).fit_transform(X)
dom = standin.get_theta().argmax(axis=1)

fig, ax = plt.subplots(figsize=(7, 6))
palette = sns.color_palette("Set2", n_colors=standin.n_topics)
for k in range(standin.n_topics):
    sel = dom == k
    label = "T%d: %s" % (k, ", ".join(standin.get_topics()[k][:2]))
    ax.scatter(coords[sel, 0], coords[sel, 1], color=palette[k], label=label, s=60)
ax.set_title("Documents in topic space (offline PCA analogue)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

## 10. Downstream prediction with a Neural Additive Model

Topic models are not only for exploration. STREAM can plug the learned topic representation straight into a **Neural Additive Model (NAM)** for an *interpretable* downstream task — e.g. predicting a satisfaction score (regression) or a churn flag (classification) from the topics a customer's tickets touch:

```python
from lightning import Trainer
from stream_topic.NAM import DownstreamModel

downstream = DownstreamModel(
    trained_topic_model=model,
    target_column="target",
    task="regression",      # or 'classification'
    dataset=dataset,
    batch_size=128,
    lr=5e-4,
)
Trainer(max_epochs=10).fit(downstream)
```

Why a NAM and not a plain MLP? A NAM learns a **separate shape function per feature** (here, per topic) and sums them. That keeps the model interpretable: you can read off *how each topic contributes* to the prediction — "more *billing* talk pushes the satisfaction score down by X" — which is exactly the kind of explanation a business stakeholder wants.

Offline we mimic the *concept* with a linear model on the `theta` features, so you can see topics-as-features actually predicting a target. We synthesise a target where the *billing* topic drags satisfaction down.

In [ ]:
if HAS_STREAM:
    print("With STREAM + lightning installed, the real flow is:")
    print("    from lightning import Trainer")
    print("    from stream_topic.NAM import DownstreamModel")
    print("    ds = DownstreamModel(trained_topic_model=model, target_column='target',")
    print("                         task='regression', dataset=dataset, batch_size=128, lr=5e-4)")
    print("    Trainer(max_epochs=10).fit(ds)\n")

# Offline analogue: topic features (theta) -> linear regression on a target.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

theta_feat = standin.get_theta()
theta_feat = theta_feat / (theta_feat.sum(axis=1, keepdims=True) + 1e-12)

# Synthesise a 'satisfaction' target: topic 0 weights chosen to illustrate.
true_weights = np.linspace(1.0, -1.0, standin.n_topics)  # last topic hurts
y = 3.0 + theta_feat @ true_weights + rng.normal(0, 0.15, size=len(DOCS))
y = np.clip(y, 1, 5)  # a 1-5 satisfaction score

reg = LinearRegression().fit(theta_feat, y)
pred = reg.predict(theta_feat)
print(f"R^2 of topics -> satisfaction: {r2_score(y, pred):.3f}")
print("\nPer-topic contribution (NAM-style readout):")
for k, w in enumerate(reg.coef_):
    sign = "raises" if w >= 0 else "lowers"
    print(f"  T{k} ({', '.join(standin.get_topics()[k][:2])}): {sign} score by {abs(w):.2f}")

## 🧪 Exercises

1. **Add a fourth theme.** Append ~8 documents about a new topic (e.g. *customer-service responsiveness*) to `DOCS`, re-run the sweep in §8, and check whether the peak NPMI moves to `k = 4`.
2. **Swap the stand-in's backend.** Replace `NMF` with `sklearn.cluster.KMeans` on the TF-IDF matrix inside `StandInTopicModel` (a clustering model, mirroring `KmeansTM`). Derive top words from each cluster centroid. Does coherence change?
3. **Coherence vs. diversity.** Extend `npmi_coherence` with a **topic-diversity** score = (unique top words across all topics) / (total top words). Plot coherence and diversity against `k` — they usually trade off.
4. **Read `get_theta` like a CRM.** For each document, print its dominant topic and second-strongest topic. Which documents are genuinely *mixed* (no single topic > 0.6)?
5. **(If you install STREAM)** Fit `KmeansTM` and `ProdLDA` on `BBC_News`, score both with `ISIM` and `NPMI`, and write two sentences on where the metrics agree and disagree.

## 🧠 Key takeaways

- **STREAM is a *unified interface*, not a single model.** Classical (`LDA`, `NMF`), neural (`ProdLDA`, `ETM`, `CTM`) and clustering (`KmeansTM`, `CEDC`) models all train via the same `fit` / `get_topics` pattern — swapping families is a one-line change, which makes honest comparison cheap.
- **Two mental models.** *Probabilistic* models give `get_beta` (P(word|topic)) and `get_theta` (P(topic|doc)) — soft mixed membership; *clustering* models assign documents to embedding clusters. Pick by your data (short noisy text → embedding/clustering; need a generative mixture → probabilistic).
- **Evaluate, don't eyeball.** Embedding metrics (`ISIM`, `INT`, `ISH`, `Expressivity`) track human judgement; `NPMI` is the classic word-cooccurrence coherence. Report both and investigate disagreements.
- **Let the tool pick `k`.** `optimize_and_fit` sweeps the topic count under an information criterion instead of guesswork.
- **Topics are features.** `DownstreamModel` (a Neural Additive Model) turns the topic representation into an *interpretable* predictor — read off how each topic moves the target.
- **Offline-first engineering matters.** Guarding every library call with `HAS_STREAM` and providing a faithful stand-in kept this whole notebook reproducible with nothing but `scikit-learn`.

## 🚀 Next step

Topics tell you *what* people are talking about. The natural next question is *how they feel* about it — move on to **[`37_sentiment_analysis.ipynb`](37_sentiment_analysis.ipynb)** to layer sentiment on top of these themes (e.g. "*billing* mentions skew negative; *delivery* is mixed"), turning a topic model into an actionable voice-of-customer dashboard.

**References**

- STREAM repository & docs: <https://github.com/AnFreTh/STREAM>
- `pip install stream-topic`

---

*Notebook 36 — Topic Modeling with STREAM · Module 9 (Optional) · Python for AI-Driven Automation and Business Data Science.*